# Preprocessing without leakage

Demonstrate grouped splitting and pipeline-contained imputers, encoders, and scalers. The held-out fold is never used to fit a transformer.

In [ ]:
from pathlib import Path
import json, sys
ROOT = Path.cwd()
if not (ROOT / 'src').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from disease_prediction.data import load_dataset
from disease_prediction.preprocessing import make_pipeline
from disease_prediction.training import _candidate_models, SEED

X, y, report = load_dataset('kidney', ROOT / 'data' / 'raw')
groups = pd.util.hash_pandas_object(X.astype('string'), index=False).astype(str)
train_idx, test_idx = next(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
print('train', X_train.shape, 'test', X_test.shape, 'training-only fitting follows')

In [ ]:
pipeline = make_pipeline(X_train, _candidate_models(y_train)['Logistic Regression'][0])
pipeline.fit(X_train, y_train)
print('transformed feature count:', len(pipeline.named_steps['preprocess'].get_feature_names_out()))
print('holdout rows transformed:', pipeline.named_steps['preprocess'].transform(X_test).shape[0])